In [1]:
import numpy as np 
import pandas as pd 

from sklearn.model_selection import train_test_split

In [3]:
df = pd.read_csv('CO2_Emissions_Canada.csv')
df.sample(5)

,Make,Model,Vehicle Class,Engine Size(L),Cylinders,Transmission,Fuel Type,Fuel Consumption City (L/100 km),Fuel Consumption Hwy (L/100 km),Fuel Consumption Comb (L/100 km),Fuel Consumption Comb (mpg),CO2 Emissions(g/km)
1794,LINCOLN,MKZ AWD,MID-SIZE,3.7,6,AS6,X,13.1,9.2,11.3,25,260
1026,VOLKSWAGEN,BEETLE TDI (modified),COMPACT,2.0,4,M6,D,8.8,6.1,7.7,37,208
7171,MERCEDES-BENZ,AMG E 53 4MATIC+ Cabriolet,SUBCOMPACT,3.0,6,A9,Z,12.5,9.0,10.9,26,245
5061,LEXUS,RX 350 L AWD,SUV - SMALL,3.5,6,AS8,X,13.1,9.4,11.1,25,268
5171,MERCEDES-BENZ,METRIS PASSENGER,SPECIAL PURPOSE VEHICLE,2.0,4,A7,Z,12.3,10.3,11.4,25,268


In [5]:
cat_cols = ['Make',	'Model', 'Vehicle Class']

for col in cat_cols:
    df[col] = df[col].str.lower()

df = df.drop_duplicates()

x = df.drop(columns=['CO2 Emissions(g/km)'])
y = df['CO2 Emissions(g/km)']
x_train, x_test, y_train, y_test = train_test_split(x, y , test_size=.2, random_state=42)

In [10]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 16.9 MB/s eta 0:00:00


In [11]:
!pip install mlflow dagshub

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.4/49.4 kB 3.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 3.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 99.8 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 101.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 71.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.1/273.1 kB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.0/212.0 kB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.9/94.9 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [12]:
import dagshub
dagshub.init(
    repo_owner='anni0955', 
    repo_name='CO2-Emission', 
    mlflow=True
)

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=11e4b49c-881d-4700-934e-c831c1c5defe&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=ad1a5f55a9bb23c620e50533c0bf61eebae1a2df87942ba57b7f3fcdc5b63c5c




Accessing as anni0955

Initialized MLflow to track repo "anni0955/CO2-Emission"

Repository anni0955/CO2-Emission initialized!

In [13]:
import mlflow 
mlflow.set_experiment('EXP 1 - Model Selection')

2026/06/15 02:55:14 INFO mlflow.tracking.fluent: Experiment with name 'EXP 1 - Model Selection' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/0950dcd8164144569083eb70f558a873', creation_time=1781492114213, effective_trace_archival_retention=None, experiment_id='0', last_update_time=1781492114213, lifecycle_stage='active', name='EXP 1 - Model Selection', tags={}, trace_location=None, workspace='default'>

In [15]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.svm import SVR
from xgboost import XGBRegressor

import optuna

In [20]:
df.sample(5)

,Make,Model,Vehicle Class,Engine Size(L),Cylinders,Transmission,Fuel Type,Fuel Consumption City (L/100 km),Fuel Consumption Hwy (L/100 km),Fuel Consumption Comb (L/100 km),Fuel Consumption Comb (mpg),CO2 Emissions(g/km)
683,lincoln,mkt awd,suv - standard,3.5,6,AS6,X,14.8,10.4,12.8,22,294
4574,chevrolet,cruze hatchback,mid-size,1.4,4,M6,X,8.6,6.2,7.5,38,175
410,ford,fusion,mid-size,1.6,4,M6,X,9.4,6.4,8.1,35,186
7204,mini,cooper s 5 door,subcompact,2.0,4,AM7,Z,8.9,6.6,7.9,36,184
4967,jeep,compass,suv - small,2.4,4,A6,X,10.6,7.6,9.3,30,218


In [23]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.preprocessing import PowerTransformer, StandardScaler

In [28]:
transformer = ColumnTransformer([
    ('ohe', OneHotEncoder(drop='first', handle_unknown='ignore'), ['Make', 'Model', 'Vehicle Class', 'Transmission', 'Fuel Type']),
    ('pt', PowerTransformer(), ['Fuel Consumption City (L/100 km)', 'Fuel Consumption Hwy (L/100 km)', 'Fuel Consumption Comb (L/100 km)', 'Fuel Consumption Comb (mpg)']),
    ('scale', StandardScaler(),['Engine Size(L)', 'Cylinders', 'Fuel Consumption City (L/100 km)', 'Fuel Consumption Hwy (L/100 km)', 'Fuel Consumption Comb (L/100 km)', 'Fuel Consumption Comb (mpg)'])
])

In [29]:
x_train_trans = transformer.fit_transform(x_train)
x_test_trans = transformer.transform(x_test)

/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


In [32]:
x_train_trans.shape

(4792, 1585)

In [33]:
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

In [46]:
def objective(trial):
    with mlflow.start_run(nested=True):
        model_name = trial.suggest_categorical('model', ['LR', 'SVR','DT', 'RF', 'GB', 'XGB'])

        if model_name == 'LR':
            model = LinearRegression()

        
        elif model_name == 'SVR':
            kernel_svm = trial.suggest_categorical('kernel_svm', ['linear', 'poly', 'rbf'])

            if kernel_svm == 'linear':
                c_linear = trial.suggest_float('c_linear', 1e-3, 10, log=True)
                model = SVR(C=c_linear, kernel='linear')

            elif kernel_svm == 'poly':
                c_poly = trial.suggest_float('c_poly', 1e-3, 10, log=True)
                degree_poly = trial.suggest_int('degree_poly', 1, 4)
                model = SVR(C=c_poly, degree=degree_poly, kernel='poly')

            else:
                c_rbf = trial.suggest_float('c_rbf', 1e-3, 10, log=True)
                gamma_rbf = trial.suggest_float('gamma_rbf', 1e-3, 10, log=True)
                model = SVR(C=c_rbf, gamma=gamma_rbf, kernel='rbf')

        elif model_name == 'DT':
            criterion = trial.suggest_categorical('criterion', ['friedman_mse', 'absolute_error', 'poisson', 'squared_error'])
            max_depth = trial.suggest_int('max_depth', 4, 20)
            model = DecisionTreeRegressor(criterion=criterion, max_depth=max_depth)

        elif model_name == 'RF':
            n_estimators = trial.suggest_int('n_estimators', 50, 500)
            max_depth = trial.suggest_int('max_depth', 4, 20)
            model = RandomForestRegressor(n_estimators=n_estimators, max_depth=max_depth, random_state=42, n_jobs=-1)
        
        elif model_name == 'GB':
            n_estimators = trial.suggest_int('n_estimators', 50, 500)
            learning_rate = trial.suggest_float('learning_rate', 1e-2, 1)
            max_depth = trial.suggest_int('max_depth', 4, 20)
            model = GradientBoostingRegressor(n_estimators=n_estimators, learning_rate=learning_rate, max_depth=max_depth, random_state=42)

        else:
            n_estimators = trial.suggest_int('n_estimators', 50, 500)
            learning_rate = trial.suggest_float('learning_rate', 1e-2, 1)
            max_depth = trial.suggest_int('max_depth', 4, 20)
            model = XGBRegressor(n_estimators=n_estimators, learning_rate=learning_rate, max_depth=max_depth, random_state=42, n_jobs=-1)    

        
        model.fit(x_train_trans, y_train)
        mlflow.log_params(trial.params)

        test_y_pred = model.predict(x_test_trans)
        train_y_pred = model.predict(x_train_trans)

        test_MAE = mean_absolute_error(y_test, test_y_pred)
        train_MAE = mean_absolute_error(y_train, train_y_pred)

        test_RMSE = root_mean_squared_error(y_test, test_y_pred)
        train_RMSE = root_mean_squared_error(y_train, train_y_pred)

        mlflow.log_metric('train_MAE', train_MAE)
        mlflow.log_metric('test_MAE', test_MAE)

        mlflow.log_metric('train_RMSE', train_RMSE)
        mlflow.log_metric('test_RMSE', test_RMSE)

        return test_RMSE


In [47]:
study = optuna.create_study(direction='minimize', study_name='Model Selection')

with mlflow.start_run(run_name='Best Model') as parent:
    study.optimize(objective, n_trials=50, n_jobs=-1)
    mlflow.log_params(study.best_params)

    mlflow.log_metric('best_score', study.best_value)

[I 2026-06-15 04:27:05,310] A new study created in memory with name: Model Selection


🏃 View run respected-elk-593 at: https://dagshub.com/anni0955/CO2-Emission.mlflow/#/experiments/0/runs/e767b0c866154062aed93301bc17cdb4
🧪 View experiment at: https://dagshub.com/anni0955/CO2-Emission.mlflow/#/experiments/0
🏃 View run thoughtful-rat-278 at: https://dagshub.com/anni0955/CO2-Emission.mlflow/#/experiments/0/runs/1905d03ca83549b2a4416a4d36538b18
🧪 View experiment at: https://dagshub.com/anni0955/CO2-Emission.mlflow/#/experiments/0


[I 2026-06-15 04:27:19,811] Trial 1 finished with value: 21.64943119968877 and parameters: {'model': 'SVR', 'kernel_svm': 'linear', 'c_linear': 0.002033317154579505}. Best is trial 1 with value: 21.64943119968877.
[I 2026-06-15 04:27:19,903] Trial 0 finished with value: 14.193096059187859 and parameters: {'model': 'SVR', 'kernel_svm': 'linear', 'c_linear': 0.1523152032837918}. Best is trial 0 with value: 14.193096059187859.


🏃 View run capable-perch-526 at: https://dagshub.com/anni0955/CO2-Emission.mlflow/#/experiments/0/runs/e72e39d5fb4b4516af82956f66f95706
🧪 View experiment at: https://dagshub.com/anni0955/CO2-Emission.mlflow/#/experiments/0


[I 2026-06-15 04:27:25,582] Trial 2 finished with value: 3.899796879593596 and parameters: {'model': 'LR'}. Best is trial 2 with value: 3.899796879593596.


...

🏃 View run youthful-cod-463 at: https://dagshub.com/anni0955/CO2-Emission.mlflow/#/experiments/0/runs/efe9828cf776422da124fd3c7eab97a9
🧪 View experiment at: https://dagshub.com/anni0955/CO2-Emission.mlflow/#/experiments/0


[I 2026-06-15 04:38:48,052] Trial 48 finished with value: 4.466131210327148 and parameters: {'model': 'XGB', 'n_estimators': 419, 'learning_rate': 0.34552915498323034, 'max_depth': 12}. Best is trial 2 with value: 3.899796879593596.


🏃 View run inquisitive-ram-902 at: https://dagshub.com/anni0955/CO2-Emission.mlflow/#/experiments/0/runs/b796be627cb24813b666845d61a8c48f
🧪 View experiment at: https://dagshub.com/anni0955/CO2-Emission.mlflow/#/experiments/0
🏃 View run wistful-snake-332 at: https://dagshub.com/anni0955/CO2-Emission.mlflow/#/experiments/0/runs/adce634faa8b46298280d367e1568c37
🧪 View experiment at: https://dagshub.com/anni0955/CO2-Emission.mlflow/#/experiments/0


[I 2026-06-15 04:38:59,475] Trial 49 finished with value: 4.519293308258057 and parameters: {'model': 'XGB', 'n_estimators': 410, 'learning_rate': 0.3640712510837467, 'max_depth': 12}. Best is trial 2 with value: 3.899796879593596.


🏃 View run Best Model at: https://dagshub.com/anni0955/CO2-Emission.mlflow/#/experiments/0/runs/28c36e2e877d4cabb3a0b8d6b4049fa4
🧪 View experiment at: https://dagshub.com/anni0955/CO2-Emission.mlflow/#/experiments/0
